In [ ]:
%%writefile /kaggle/working/hpo.py
"""
Phase 1 - hyperparameter optimisation (Optuna TPE = Bayesian optimisation + median pruning).
Short runs (5 epochs) on the stratified 30% HPO subset, scored on the full validation set.
Launched by the next notebook cell; one process per GPU, all sharing one SQLite study.
"""
import argparse, glob, json, math, os, random, time

import numpy as np
import optuna
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

INPUT_ROOT = "/kaggle/input"
OUT_DIR = "/kaggle/working"
IMG_SIZE = 224  # same input size for both models, as in the paper

# Paper configurations (Bahaa et al., AgriEngineering 2026) - enqueued as trial 0 per model.
# Adam without weight decay ~ AdamW with the smallest weight decay in the search space.
# Head dropout is not stated in the paper -> 0.2 for MobileNet (paper mentions dropout), 0.0 for MobileViT.
PAPER_CONFIGS = {
    "mobilenetv3_small_100": {"lr": 1e-3, "weight_decay": 1e-6, "batch_size": 64,
                              "dropout": 0.2, "label_smoothing": 0.0, "schedule": "constant"},
    "mobilevit_xxs":         {"lr": 1e-4, "weight_decay": 1e-6, "batch_size": 32,
                              "dropout": 0.0, "label_smoothing": 0.0, "schedule": "constant"},
}


def find_image_root(sample_path):
    """Return the folder that the relative paths in splits.csv start from.
    Works whether paths look like 'color/...' or '<dataset>/.../color/...'."""
    if os.path.exists(os.path.join(INPUT_ROOT, sample_path)):
        return INPUT_ROOT
    first = sample_path.split("/")[0]  # e.g. 'color'
    for d in glob.glob(f"{INPUT_ROOT}/**/{first}", recursive=True):
        root = os.path.dirname(d)
        if os.path.exists(os.path.join(root, sample_path)):
            return root
    raise FileNotFoundError(f"Could not find images for {sample_path} - attach the PlantVillage dataset.")


class LeafDataset(Dataset):
    def __init__(self, df, tf, root):
        self.paths = [os.path.join(root, p) for p in df["path"]]
        self.labels = df["label_idx"].tolist()
        self.tf = tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), self.labels[i]


def build_transforms(model_name):
    """Augmentation from the paper (h/v flips, rotation +-10 deg, brightness 0.5-1.5).
    Normalisation uses each model's own pretraining statistics from timm:
    ImageNet mean/std for MobileNetV3, plain [0, 1] scaling for MobileViT (cvnets weights)."""
    cfg = timm.models.get_pretrained_cfg(model_name).to_dict()
    mean, std = cfg["mean"], cfg["std"]
    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=(0.5, 1.5)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    return train_tf, eval_tf, mean, std


def make_loader(df, tf, batch_size, shuffle, workers, seed, root):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(LeafDataset(df, tf, root), batch_size=batch_size, shuffle=shuffle,
                      num_workers=workers, pin_memory=True, persistent_workers=workers > 0,
                      generator=g, drop_last=shuffle)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def build_scheduler(opt, schedule, steps_per_epoch, epochs):
    total, warmup = steps_per_epoch * epochs, steps_per_epoch  # 1 warmup epoch for cosine

    def fn(step):
        if schedule == "constant":
            return 1.0
        if step < warmup:
            return (step + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))

    return torch.optim.lr_scheduler.LambdaLR(opt, fn)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ce = nn.CrossEntropyLoss(reduction="sum")
    total, preds, labels = 0.0, [], []
    for x, y in loader:
        x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
        with torch.amp.autocast("cuda"):
            out = model(x)
        total += ce(out.float(), y).item()
        preds.append(out.argmax(1).cpu()); labels.append(y.cpu())
    preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
    return {"loss": total / len(labels), "acc": float((preds == labels).mean()),
            "macro_f1": float(f1_score(labels, preds, average="macro"))}


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", default="mobilenetv3_small_100")
    p.add_argument("--n-trials", type=int, default=20)
    p.add_argument("--epochs", type=int, default=5)
    p.add_argument("--workers", type=int, default=4)
    p.add_argument("--seed", type=int, default=42)
    args = p.parse_args()
    torch.backends.cudnn.benchmark = True

    found = glob.glob(f"{INPUT_ROOT}/**/splits.csv", recursive=True)
    assert found, "splits.csv not found - attach your splits dataset."
    df = pd.read_csv(found[0])
    num_classes = df["label_idx"].nunique()
    train_df = df[(df.split == "train") & (df.hpo_subset)]
    val_df = df[df.split == "val"]
    img_root = find_image_root(df["path"].iloc[0])
    missing = [p for p in df["path"].sample(200, random_state=0)
               if not os.path.exists(os.path.join(img_root, p))]
    assert not missing, f"{len(missing)}/200 sampled images missing under {img_root}"
    TRAIN_TF, EVAL_TF, mean, std = build_transforms(args.model)
    print(f"GPU: {torch.cuda.get_device_name(0)} | model: {args.model} | normalise mean={mean} "
          f"std={std} | images: {img_root} | HPO: {len(train_df)} train / {len(val_df)} val",
          flush=True)

    study = optuna.create_study(
        study_name=args.model, storage=f"sqlite:///{OUT_DIR}/hpo_{args.model}.db",
        direction="maximize", load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=args.seed, n_startup_trials=5,
                                           multivariate=True, constant_liar=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1))
    if len(study.trials) == 0:
        study.enqueue_trial(PAPER_CONFIGS[args.model])  # trial 0 = paper's configuration

    def objective(trial):
        cfg = {
            "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-1, log=True),
            "batch_size": trial.suggest_categorical("batch_size", [32, 64]),
            "dropout": trial.suggest_categorical("dropout", [0.0, 0.2, 0.5]),
            "label_smoothing": trial.suggest_categorical("label_smoothing", [0.0, 0.1]),
            "schedule": trial.suggest_categorical("schedule", ["constant", "cosine"]),
        }
        print(f"\n=== Trial {trial.number}: {cfg}", flush=True)
        set_seed(args.seed)
        model = timm.create_model(args.model, pretrained=True, num_classes=num_classes,
                                  drop_rate=cfg["dropout"]).cuda()
        train_loader = make_loader(train_df, TRAIN_TF, cfg["batch_size"], True, args.workers,
                                   args.seed, img_root)
        val_loader = make_loader(val_df, EVAL_TF, 128, False, args.workers, args.seed, img_root)
        opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
        sched = build_scheduler(opt, cfg["schedule"], len(train_loader), args.epochs)
        scaler = torch.amp.GradScaler("cuda")
        criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])

        history, best_f1, t_start = [], -1.0, time.time()
        for epoch in range(args.epochs):
            t0 = time.time()
            model.train()
            tr_loss, n = 0.0, 0
            for x, y in train_loader:
                x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda"):
                    loss = criterion(model(x), y)
                scaler.scale(loss).backward()
                scaler.step(opt); scaler.update(); sched.step()
                tr_loss += loss.item() * x.size(0); n += x.size(0)
            val = evaluate(model, val_loader)
            row = {"trial": trial.number, "epoch": epoch + 1, "train_loss": tr_loss / n,
                   **{f"val_{k}": v for k, v in val.items()}, "epoch_time_s": time.time() - t0}
            history.append(row)
            print(json.dumps(row), flush=True)
            best_f1 = max(best_f1, val["macro_f1"])

            trial.report(val["macro_f1"], epoch)
            if trial.should_prune():
                trial.set_user_attr("history", history)
                print(f"Trial {trial.number} pruned", flush=True)
                raise optuna.TrialPruned()

        trial.set_user_attr("history", history)
        trial.set_user_attr("train_time_s", time.time() - t_start)
        return best_f1

    # Stop when the whole study (all GPUs together) has n_trials finished trials
    stop = optuna.study.MaxTrialsCallback(
        args.n_trials, states=(optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED))
    study.optimize(objective, n_trials=args.n_trials, callbacks=[stop], gc_after_trial=True)


if __name__ == "__main__":
    main()


In [ ]:
# ---- Launcher: runs one HPO worker per GPU, waits for both, then summarises ----
import json, os, subprocess, sys, threading, time
import torch

MODEL = "mobilenetv3_small_100"   # later: "mobilevit_xxs"
N_TRIALS = 20

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=False)

n_gpus = torch.cuda.device_count()
assert n_gpus > 0, "No GPU - set Accelerator to GPU T4 x2."
workers = 4 if n_gpus == 1 else 2   # Kaggle has 4 CPU cores in total
print(f"{n_gpus} GPU(s), {workers} data workers each", flush=True)


def stream(proc, tag):
    for line in proc.stdout:
        print(f"[{tag}] {line}", end="", flush=True)


procs = []
for g in range(n_gpus):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    proc = subprocess.Popen(
        [sys.executable, "-u", "/kaggle/working/hpo.py", "--model", MODEL,
         "--n-trials", str(N_TRIALS), "--workers", str(workers)],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    t = threading.Thread(target=stream, args=(proc, f"gpu{g}"), daemon=True)
    t.start()
    procs.append((proc, t))
    if g == 0 and n_gpus > 1:
        time.sleep(90)  # let worker 0 create the study and take trial 0 first

for proc, t in procs:
    proc.wait()
    t.join()
codes = [proc.returncode for proc, _ in procs]
print("Worker exit codes:", codes)
assert all(c == 0 for c in codes), "A worker failed - see its log lines above."

# ---- Summary ----
import optuna
study = optuna.load_study(study_name=MODEL, storage=f"sqlite:////kaggle/working/hpo_{MODEL}.db")
df = study.trials_dataframe()
df.to_csv(f"/kaggle/working/hpo_{MODEL}_trials.csv", index=False)

done = sorted([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE],
              key=lambda t: t.value, reverse=True)
n_pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in study.trials)
print(f"\n{len(done)} complete, {n_pruned} pruned trials")
paper = study.trials[0]
print(f"Paper config (trial 0): macro-F1 = {paper.value}")
print("\nTop 5:")
for rank, t in enumerate(done[:5], 1):
    print(f"  {rank}. trial {t.number}: macro-F1 {t.value:.4f}  {t.params}")

with open(f"/kaggle/working/hpo_{MODEL}_top3.json", "w") as f:
    json.dump([{"rank": r, "trial": t.number, "macro_f1": t.value, "params": t.params}
               for r, t in enumerate(done[:3], 1)], f, indent=2)
print("\nSaved: hpo_*.db, hpo_*_trials.csv, hpo_*_top3.json")
